In [2]:
"""
01_boundary_mass_diagnostic.py   (v2)
=====================================
Quantifies how much probability mass sits beyond the +/-5 clip boundary in the
aggregate model-ready datasets, per feature and in aggregate.

CHANGES FROM v1 (all of these altered the conclusions, not just the presentation)
--------------------------------------------------------------------------------
1. Moment-type detection no longer collides with macro features whose NAMES end
   in "_spread" (other_spread, lev_spread, bull_bear_spread, bbb_aaa_spread,
   vix_term_spread, ff_2y_spread, ...). v1 counted ~11-13 raw macro series as
   "spread moments". A suffix is now only treated as a moment if the same base
   factor also appears with _cwmean, which is true for genuine moment columns
   and false for a macro series that merely ends in the word "spread".

2. Classifier order fixed. v1 tested "nonstationary_trend" before
   "artefact_spike", and a one-point spike trivially passes the trend test
   (sign_skew = 1.0 by construction; late_share is 0 or 1 with n=1). Result:
   every top-magnitude artefact (ivol_q, dollarrealizedspread_lr_dw,
   effectivespread_dollar_dw, venue_range_a) was labelled a trend. Artefact is
   now tested first and the trend test requires a minimum exceedance count.

3. "rare_event" removed. v1's rule (unique_ratio < 0.15 and few runs) fired on
   EVERY monthly feature, because a monthly series forward-filled to daily has
   unique_ratio ~ 0.048 regardless of its value distribution. It therefore
   labelled monthly_m1, monthly_saving_rate and monthly_personal_income_mom as
   rare events. Zero-inflation is a property of the RAW value distribution and
   cannot be recovered from z-scores, so it is now measured on Stage 2 raw data
   (modal share, zero share) and reported as such.

4. Mass concentration added. v1's verdict asserted "N features carry most of the
   mass" without computing it. Top-N share of total exceedance mass is now
   measured directly, because it decides whether per-feature fixes suffice.

5. Raw-data enrichment added from Stage 2 (pre-z-score). Gives, per feature:
   modal share, zero share, robust scale sigma_f, and the ratio std/sigma_f --
   the direct empirical test of the "tiny standard deviation" hypothesis. A
   large ratio means the std is INFLATED by outliers (which SUPPRESSES z-scores,
   the opposite of the usual worry); a ratio near or below 1 with high wall mass
   means the numerator is genuinely large.

WHERE THE DATA COMES FROM
-------------------------
Stage 4 holds the last UNCLIPPED z-scores:
    Stage 3              z-score, drop warmup, save.        No clip.
    CALENDAR_REMOVAL     drop 8 calendar cols.              No clip.
    01_prepare_datasets  CLIP_LIMIT = 5.0 applied here.     <-- clip enters
Stage 2 holds the raw pre-z-score values, used only for the enrichment above.

DEDUPLICATION
-------------
Monthly features are forward-filled ~21x. Counting exceedances on daily rows
multiplies their apparent severity by that factor. Counts are reported raw and
deduplicated; deduplicated drives everything.

NOTE ON THE WEEKLY FEATURES
---------------------------
In Stage 4 the 34 weekly features have unique_ratio = 1.000, i.e. their z-score
changes every single day. That is not a measurement error -- it IS the original
bug: a constant numerator over a moving expanding mean/std produces a new z
every day. Consequently dedup cannot compress them in Stage 4, and the global
figure is reported a second time with the weekly-fixed values substituted, so
"what the model actually receives" is visible alongside "what Stage 4 contains".
"""

import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

PROJECT_ROOT = Path("../..")

STAGE4_DIR = PROJECT_ROOT / "Data/Data_Collection/Final/Stage_4_Final_w_Calendar_Theme_Removed"
STAGE2_DIR = PROJECT_ROOT / "Data/Data_Collection/Final/Stage_2"
PANEL_C_PATH = (PROJECT_ROOT / "Data/Data_Collection/Final"
                / "Stage_1_5_Validation_and_Feature_Engineering"
                / "panel_macro_daily_engineered.parquet")

THEME_DIR_PRIMARY = PROJECT_ROOT / "Data/Splits/themes"
THEME_DIR_FALLBACK = STAGE4_DIR / "themes"

OUTPUT_DIR = PROJECT_ROOT / "Data/Diagnostics"

THRESHOLDS = [3.0, 5.0, 10.0, 50.0]
CLIP_LIMIT = 5.0

# Assumed KAN grid resolution over [-CLIP_LIMIT, +CLIP_LIMIT], used only to
# express clipped mass as a share of the outermost spline bin's population.
# Change to match the actual grid if it differs; it affects one report line.
KAN_GRID_POINTS = 8

DATASETS = {
    "means_only": {
        "parquet": "model_market_combined_means.parquet",
        "theme_csv": "combined_means_theme_assignment.csv",
        "stage2_daily": "agg_market_daily_means.parquet",
        "stage2_monthly": "agg_market_monthly_means.parquet",
        "has_moments": False,
    },
    "full_moments": {
        "parquet": "model_market_combined_full_moments.parquet",
        "theme_csv": "combined_full_moments_theme_assignment.csv",
        "stage2_daily": "agg_market_daily_full_moments.parquet",
        "stage2_monthly": "agg_market_monthly_full_moments.parquet",
        "has_moments": True,
    },
}

META_COLS = {"date", "target_daily_return", "target_monthly_return",
             "minret_5d", "minret_5d_z", "y_binary"}

BINARY_FEATURES = [
    "vix_above_20", "vix_above_30",
    "curve_inverted_2y10y", "curve_inverted_3m10y", "credit_stress",
]

MOMENT_SUFFIXES = ["_cwmean", "_cwstd", "_cwskew", "_cwkurt", "_spread"]

H41_DELAY_FEATURES = ["fed_assets", "tga", "reserves"]
WEEKLY_FEATURES = sorted(set([
    "lev_long", "lev_short", "lev_spread",
    "am_long", "am_short", "am_spread",
    "dealer_long", "dealer_short", "dealer_spread",
    "other_long", "other_short", "other_spread",
    "open_interest",
    "lev_net", "am_net", "dealer_net",
    "lev_net_pct", "am_net_pct", "dealer_net_pct",
    "lev_am_ratio", "lev_net_chg", "am_net_chg",
    "bullish", "neutral", "bearish", "bullish_8w_ma", "bull_bear_spread",
    "initial_claims", "continued_claims",
    "fed_assets", "tga", "reserves",
    "bank_credit", "ci_loans",
]))
WEEKLY_Z_MIN_PERIODS = 52

# ── Classification thresholds ────────────────────────────────────────────────
# An expanding z-score over 300+ observations cannot legitimately reach |z| in
# the hundreds: that requires either a near-zero denominator or a numerator
# wrong by orders of magnitude. 100 is comfortably past anything a real
# financial series produces.
ARTEFACT_MAX_Z = 100.0
ARTEFACT_N_MAX = 15
# Trend requires enough exceedances for concentration and sign statistics to
# mean anything. With n_exc = 1, sign_skew is 1.0 and late_share is 0 or 1 by
# construction -- which is precisely how v1 mislabelled every artefact.
TREND_MIN_EXC = 15
TREND_SPEARMAN_MIN = 0.40
TREND_SIGN_SKEW_MIN = 0.80
TREND_LATE_SHARE_MIN = 0.60
HIGH_MASS_MIN = 0.01
ZERO_INFLATED_MODAL_MIN = 0.50
STD_INFLATION_MIN = 20.0


# ═══════════════════════════════════════════════════════════════════════════════
# WEEKLY Z-SCORE RECONSTRUCTION
# ═══════════════════════════════════════════════════════════════════════════════

def build_weekly_zscored_daily(panel_c_path, weekly_features, h41_delay,
                               min_periods=WEEKLY_Z_MIN_PERIODS):
    """
    Rebuild weekly features z-scored at WEEKLY frequency, then forward-filled.
    Reproduced from 01_prepare_datasets Phase 1.5 so this measures exactly what
    the model receives.
    """
    cols_needed = ["date"] + weekly_features
    panel_c = pd.read_parquet(panel_c_path, columns=cols_needed)
    panel_c["date"] = pd.to_datetime(panel_c["date"])
    panel_c = panel_c.sort_values("date").reset_index(drop=True)
    n_days = len(panel_c)

    result = pd.DataFrame({"date": panel_c["date"]})
    n_updates_log = {}

    for col_name in weekly_features:
        raw = panel_c[col_name].values.astype(np.float64).copy()

        is_update = np.zeros(n_days, dtype=bool)
        is_update[0] = True
        is_update[1:] = np.abs(np.diff(raw)) > 1e-12

        update_idx = np.where(is_update)[0]
        update_vals = raw[update_idx]
        n_updates = len(update_vals)
        n_updates_log[col_name] = n_updates

        z_at_updates = np.full(n_updates, np.nan)
        for k in range(min_periods, n_updates):
            past = update_vals[:k]
            mu = np.mean(past)
            sigma = np.std(past, ddof=1)
            if sigma > 1e-12:
                z_at_updates[k] = (update_vals[k] - mu) / sigma

        daily_z = np.full(n_days, np.nan)
        for k in range(n_updates):
            start = update_idx[k]
            end = update_idx[k + 1] if k + 1 < n_updates else n_days
            daily_z[start:end] = z_at_updates[k]

        if col_name in h41_delay:
            daily_z = np.roll(daily_z, 1)
            daily_z[0] = np.nan

        result[col_name] = daily_z

    return result, n_updates_log


# ═══════════════════════════════════════════════════════════════════════════════
# PER-FEATURE MEASUREMENT
# ═══════════════════════════════════════════════════════════════════════════════

def _spearman(a, b):
    if len(a) < 3:
        return np.nan
    ra = pd.Series(a).rank().values
    rb = pd.Series(b).rank().values
    if np.std(ra) < 1e-12 or np.std(rb) < 1e-12:
        return np.nan
    return float(np.corrcoef(ra, rb)[0, 1])


def _run_lengths(flags):
    if not flags.any():
        return np.array([], dtype=int)
    padded = np.concatenate(([False], flags, [False]))
    edges = np.diff(padded.astype(np.int8))
    return np.where(edges == -1)[0] - np.where(edges == 1)[0]


def measure_feature(z_raw, dates, name):
    """
    Measure boundary behaviour for one feature's z-score series.

    Counts appear twice: *_raw over non-NaN daily rows, *_dedup over genuine
    observations (rows where the value changed). Deduped drives everything,
    because a monthly feature forward-filled 21x would otherwise read as 21
    separate exceedances for one bad month.
    """
    valid = ~np.isnan(z_raw)
    n_raw = int(valid.sum())
    if n_raw == 0:
        return None

    zv = z_raw[valid]
    dv = dates[valid]
    abs_zv = np.abs(zv)

    changed = np.ones(len(zv), dtype=bool)
    changed[1:] = np.abs(np.diff(zv)) > 1e-12
    n_dedup = int(changed.sum())
    zd = zv[changed]
    dd = dv[changed]
    abs_zd = np.abs(zd)

    rec = {
        "column": name,
        "n_obs_raw": n_raw,
        "n_obs_dedup": n_dedup,
        "unique_ratio": round(n_dedup / n_raw, 4),
        "max_abs_z": float(abs_zv.max()),
        "date_of_max": str(pd.Timestamp(dv[int(np.argmax(abs_zv))]).date()),
        "p99_abs_z": float(np.percentile(abs_zd, 99)),
        "p999_abs_z": float(np.percentile(abs_zd, 99.9)),
        "mean_abs_z": float(abs_zd.mean()),
        "std_z": float(zd.std(ddof=1)) if n_dedup > 1 else 0.0,
    }

    for t in THRESHOLDS:
        tag = str(t).replace(".0", "")
        rec[f"n_gt{tag}_raw"] = int((abs_zv > t).sum())
        rec[f"n_gt{tag}_dedup"] = int((abs_zd > t).sum())
        rec[f"pct_gt{tag}_dedup"] = float((abs_zd > t).mean())

    wall = abs_zd > CLIP_LIMIT
    rec["wall_mass"] = float(wall.mean())
    rec["wall_mass_raw"] = float((abs_zv > CLIP_LIMIT).mean())

    runs_raw = _run_lengths(abs_zv > CLIP_LIMIT)
    runs_ded = _run_lengths(wall)
    rec["n_runs_raw"] = int(len(runs_raw))
    rec["max_run_raw"] = int(runs_raw.max()) if len(runs_raw) else 0
    rec["n_runs_dedup"] = int(len(runs_ded))
    rec["max_run_dedup"] = int(runs_ded.max()) if len(runs_ded) else 0

    n_exc = int(wall.sum())
    if n_exc > 0:
        n_pos = int((zd[wall] > 0).sum())
        rec["n_gt5_pos"] = n_pos
        rec["n_gt5_neg"] = n_exc - n_pos
        rec["sign_skew"] = max(n_pos, n_exc - n_pos) / n_exc
        rec["exc_dates_first3"] = ", ".join(
            str(pd.Timestamp(d).date()) for d in dd[wall][:3]
        )
    else:
        rec["n_gt5_pos"] = 0
        rec["n_gt5_neg"] = 0
        rec["sign_skew"] = np.nan
        rec["exc_dates_first3"] = ""

    t_idx = np.arange(n_dedup, dtype=np.float64)
    rec["late_share"] = (float((t_idx[wall] >= 0.75 * n_dedup).mean())
                         if n_exc > 0 else np.nan)
    rec["spearman_z_time"] = _spearman(zd, t_idx)

    return rec


# ═══════════════════════════════════════════════════════════════════════════════
# RAW-DATA ENRICHMENT FROM STAGE 2
# ═══════════════════════════════════════════════════════════════════════════════

def robust_scale(vals):
    """Scaled MAD. Quantile-only, so an extreme value cannot move it."""
    v = vals[~np.isnan(vals)]
    if len(v) < 2:
        return np.nan
    return float(1.4826 * np.median(np.abs(v - np.median(v))))


def enrich_from_raw(stats, cfg):
    """
    Attach raw-value statistics from Stage 2 (pre-z-score).

    Three things become measurable that z-scores alone cannot show:

      modal_share / zero_share
          Zero-inflation. A feature that is 0 in >50% of observations has a
          degenerate scale and is the genuine case for a denominator floor.
          This CANNOT be inferred from z-scores, because an expanding z-score
          turns a constant raw series into a varying z (the mean and std move
          underneath it). v1 tried to infer it from observation frequency and
          consequently labelled every monthly feature a rare event.

      std_over_sigma_f
          The direct test of the "tiny standard deviation" hypothesis.
            >> 1  the ordinary std is INFLATED by outliers relative to the
                  robust scale. This SUPPRESSES z-scores -- the denominator is
                  too big -- so extremes here are real numerator events, and a
                  floor would do nothing.
            ~= 1  well-behaved; std and robust scale agree.
            <  1  unusual; suggests a heavy centre with thin tails.
          A large ratio combined with high wall mass means the numerator is
          genuinely enormous, which points upstream at the feature definition
          rather than at the normalisation.
    """
    daily_p = STAGE2_DIR / cfg["stage2_daily"]
    monthly_p = STAGE2_DIR / cfg["stage2_monthly"]
    if not daily_p.exists() or not monthly_p.exists():
        print(f"    Stage 2 files absent; skipping raw enrichment")
        for c in ("raw_modal_share", "raw_zero_share", "raw_sigma_f",
                  "raw_std", "std_over_sigma_f"):
            stats[c] = np.nan
        return stats

    d2 = pd.read_parquet(daily_p)
    m2 = pd.read_parquet(monthly_p)
    print(f"    Stage 2 raw: daily {d2.shape}, monthly {m2.shape}")

    out = {}
    n_hit = 0
    for col in stats["column"]:
        if col.startswith("monthly_"):
            src, key = m2, col[len("monthly_"):]
        else:
            src, key = d2, col
        if key not in src.columns:
            out[col] = (np.nan,) * 5
            continue
        v = src[key].values.astype(np.float64)
        v = v[~np.isnan(v)]
        if len(v) < 10:
            out[col] = (np.nan,) * 5
            continue
        n_hit += 1
        vals, counts = np.unique(np.round(v, 12), return_counts=True)
        modal = float(counts.max() / len(v))
        zero = float(np.mean(np.abs(v) < 1e-12))
        sf = robust_scale(v)
        sd = float(np.std(v, ddof=1))
        ratio = sd / sf if (sf is not None and sf > 1e-12) else np.nan
        out[col] = (modal, zero, sf, sd, ratio)

    stats["raw_modal_share"] = stats["column"].map(lambda c: out[c][0])
    stats["raw_zero_share"] = stats["column"].map(lambda c: out[c][1])
    stats["raw_sigma_f"] = stats["column"].map(lambda c: out[c][2])
    stats["raw_std"] = stats["column"].map(lambda c: out[c][3])
    stats["std_over_sigma_f"] = stats["column"].map(lambda c: out[c][4])
    print(f"    Matched {n_hit} / {len(stats)} features to Stage 2 raw values")
    return stats


# ═══════════════════════════════════════════════════════════════════════════════
# ANNOTATION
# ═══════════════════════════════════════════════════════════════════════════════

def annotate(stats, theme_df, all_cols, has_moments):
    """
    Attach moment type, frequency, risk class and theme.

    Moment detection guards against the name collision that broke v1: a suffix
    is only a moment if the same base factor also appears with _cwmean. Genuine
    moment columns always come in a family of five, whereas a macro series that
    merely ends in the word "spread" (other_spread, bull_bear_spread,
    bbb_aaa_spread, vix_term_spread, ff_2y_spread, ...) has no _cwmean sibling.
    """
    colset = set(all_cols)

    def split_moment(c):
        if not has_moments:
            return c, "cwmean_unsuffixed"
        for s in MOMENT_SUFFIXES:
            if c.endswith(s):
                base = c[: -len(s)]
                if f"{base}_cwmean" in colset:
                    return base, s[1:]
                break
        return c, "raw_level"

    parsed = stats["column"].map(split_moment)
    stats["base_factor"] = [p[0] for p in parsed]
    stats["moment_type"] = [p[1] for p in parsed]

    wk = set(WEEKLY_FEATURES)

    def freq_of(base):
        if base.startswith("monthly_"):
            return "monthly"
        return "weekly" if base in wk else "daily"
    stats["declared_freq"] = stats["base_factor"].map(freq_of)

    if theme_df is not None and "panel" in theme_df.columns:
        pm = theme_df.set_index("column")["panel"].to_dict()
        stats["panel"] = stats["column"].map(pm)
        is_macro = stats["panel"].fillna("").str.contains("C|D", regex=True)
    else:
        stats["panel"] = ""
        is_macro = stats["moment_type"] == "raw_level"

    # THE SPLIT THAT MATTERS. Not aggregate-vs-panel, but whether a
    # cross-section was averaged away. A cwmean over ~100 winsorised stocks is
    # heavily variance-reduced. A macro series has no cross-section at all, so
    # it sits in the same risk class as a raw panel feature.
    stats["risk_class"] = np.where(is_macro, "no_cross_section",
                                   "cross_section_averaged")

    if theme_df is not None:
        for c in ("theme_id", "theme_name", "subtheme_id", "subtheme_name"):
            if c in theme_df.columns:
                stats[c] = stats["column"].map(
                    theme_df.set_index("column")[c].to_dict())
    return stats


# ═══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATION
# ═══════════════════════════════════════════════════════════════════════════════

def classify(r):
    """
    Assign a cause and action.

    Order is load-bearing. Artefact is tested FIRST: |z| > 100 with a handful of
    points cannot be anything else, and in v1 putting trend first swallowed all
    of them (a one-point spike is one-sided and "late" by construction).
    Zero-inflation is tested on raw values, not on observation frequency.
    Trend requires a minimum exceedance count so its sign and concentration
    statistics are not degenerate.
    """
    n_exc = r["n_gt5_dedup"]
    if n_exc == 0:
        return "clean", "keep"

    max_z = r["max_abs_z"]
    ratio = r.get("std_over_sigma_f", np.nan)

    # 1. Artefact. Unambiguous by magnitude.
    if max_z > ARTEFACT_MAX_Z and n_exc <= ARTEFACT_N_MAX:
        return "artefact_spike", "investigate_or_drop"

    # 2. Outlier-inflated scale. The std is many times the robust scale, so the
    #    denominator is too large and z-scores are being SUPPRESSED. Any
    #    exceedance that survives that is a genuinely enormous numerator, which
    #    is an upstream feature-definition problem, not a normalisation one.
    if (not np.isnan(ratio)) and ratio > STD_INFLATION_MIN and max_z > 20:
        return "outlier_driven_scale", "investigate_or_drop"

    # 3. Zero-inflated. Measured on raw values. The genuine case for a floor.
    modal = r.get("raw_modal_share", np.nan)
    if (not np.isnan(modal)) and modal > ZERO_INFLATED_MODAL_MIN:
        return "zero_inflated", "drop_or_floor"

    # 4. Non-stationary drift. Monotone in time AND one-sided AND enough points
    #    for those statistics to be meaningful. A crisis fires both tails and is
    #    not monotone, so it will not be caught here.
    if n_exc >= TREND_MIN_EXC:
        sp = r["spearman_z_time"]
        skew = r["sign_skew"]
        late = r["late_share"]
        monotone = (not np.isnan(sp)) and abs(sp) > TREND_SPEARMAN_MIN
        one_sided = (not np.isnan(skew)) and skew > TREND_SIGN_SKEW_MIN
        late_conc = (not np.isnan(late)) and late > TREND_LATE_SHARE_MIN
        if one_sided and (monotone or late_conc):
            return "nonstationary_trend", "difference_or_roll"

    # 5. Material mass with no identified mechanism.
    if r["wall_mass"] >= HIGH_MASS_MIN:
        return "high_mass_unexplained", "inspect_individually"

    return "minor_tail", "keep"


# ═══════════════════════════════════════════════════════════════════════════════
# DATASET ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

def load_theme_map(name):
    for d in (THEME_DIR_PRIMARY, THEME_DIR_FALLBACK):
        p = d / name
        if p.exists():
            print(f"    Theme CSV: {p}")
            return pd.read_csv(p)
    print(f"    WARNING: {name} not found; panel/theme grouping blank")
    return None


def analyse_dataset(name, cfg, weekly_z):
    print(f"\n{'=' * 78}")
    print(f"DATASET: {name}")
    print(f"{'=' * 78}")

    pq = STAGE4_DIR / cfg["parquet"]
    if not pq.exists():
        print(f"  MISSING: {pq}")
        return None

    df = pd.read_parquet(pq)
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)
    print(f"  {pq.name}: {df.shape[0]:,} rows x {df.shape[1]:,} cols")
    print(f"  Dates: {df['date'].min().date()} -> {df['date'].max().date()}")

    theme_df = load_theme_map(cfg["theme_csv"])
    dates = df["date"].values

    feats = [c for c in df.columns
             if c not in META_COLS and c not in BINARY_FEATURES]
    print(f"  Features analysed: {len(feats):,}")

    print(f"  Measuring Stage 4 z-scores...")
    recs = [measure_feature(df[c].values.astype(np.float64), dates, c)
            for c in feats]
    stats = pd.DataFrame([r for r in recs if r is not None])

    wk_present = [c for c in feats if c in weekly_z.columns]
    if wk_present:
        print(f"  Re-measuring {len(wk_present)} weekly features at weekly "
              f"frequency...")
        wz = df[["date"]].merge(weekly_z[["date"] + wk_present],
                               on="date", how="left")
        fixed = []
        for c in wk_present:
            r = measure_feature(wz[c].values.astype(np.float64), dates, c)
            if r is not None:
                fixed.append({
                    "column": c,
                    "wf_max_abs_z": r["max_abs_z"],
                    "wf_n_obs_dedup": r["n_obs_dedup"],
                    "wf_n_gt5_dedup": r["n_gt5_dedup"],
                    "wf_n_gt5_raw": r["n_gt5_raw"],
                    "wf_wall_mass": r["wall_mass"],
                })
        if fixed:
            stats = stats.merge(pd.DataFrame(fixed), on="column", how="left")

    stats = annotate(stats, theme_df, list(df.columns), cfg["has_moments"])
    print(f"  Enriching from Stage 2 raw values...")
    stats = enrich_from_raw(stats, cfg)

    lab = stats.apply(lambda r: classify(r.to_dict()), axis=1)
    stats["cause"] = [l[0] for l in lab]
    stats["action"] = [l[1] for l in lab]

    return stats.sort_values("wall_mass", ascending=False).reset_index(drop=True)


# ═══════════════════════════════════════════════════════════════════════════════
# REPORTING
# ═══════════════════════════════════════════════════════════════════════════════

def _band(w):
    if w == 0:
        return "0  (clean)"
    for hi, lab in [(0.001, "1  <0.1%"), (0.005, "2  0.1-0.5%"),
                    (0.01, "3  0.5-1%"), (0.02, "4  1-2%"),
                    (0.05, "5  2-5%")]:
        if w < hi:
            return lab
    return "6  >5%"


def report(name, stats, cfg):
    print(f"\n{'=' * 78}")
    print(f"REPORT: {name}")
    print(f"{'=' * 78}")

    n_feat = len(stats)
    cells_r = stats["n_obs_raw"].sum()
    cells_d = stats["n_obs_dedup"].sum()
    exc_r = stats["n_gt5_raw"].sum()
    exc_d = stats["n_gt5_dedup"].sum()

    print(f"\n1. GLOBAL BOUNDARY MASS (|z| > {CLIP_LIMIT})")
    print(f"   Raw (daily rows):    {exc_r:>9,} / {cells_r:>11,} "
          f"= {exc_r / cells_r * 100:7.4f}%")
    print(f"   Deduped (true obs):  {exc_d:>9,} / {cells_d:>11,} "
          f"= {exc_d / cells_d * 100:7.4f}%   <-- honest")

    # Substitute the weekly-fixed values to show what the model receives.
    if "wf_n_gt5_dedup" in stats.columns:
        wf = stats["wf_n_gt5_dedup"].notna()
        adj_exc = (exc_d - stats.loc[wf, "n_gt5_dedup"].sum()
                   + stats.loc[wf, "wf_n_gt5_dedup"].sum())
        adj_cells = (cells_d - stats.loc[wf, "n_obs_dedup"].sum()
                     + stats.loc[wf, "wf_n_obs_dedup"].sum())
        print(f"   With weekly fix:     {adj_exc:>9,.0f} / {adj_cells:>11,.0f} "
              f"= {adj_exc / adj_cells * 100:7.4f}%   <-- model input")

    print(f"\n   By threshold (deduped):")
    for t in THRESHOLDS:
        tag = str(t).replace(".0", "")
        e = stats[f"n_gt{tag}_dedup"].sum()
        print(f"     |z| > {t:>5}: {e:>9,} = {e / cells_d * 100:7.4f}%")

    # Clipped mass as a share of the outermost spline bin's population. This is
    # the operative quantity for the stated worry: if the outer bin already
    # contains many unclipped points, the clipped clump is a minority of it.
    bin_w = 2.0 * CLIP_LIMIT / (KAN_GRID_POINTS - 1)
    outer_lo = CLIP_LIMIT - bin_w
    outer_mass = 0
    for c in stats["column"]:
        pass  # per-feature outer mass computed below from the p-columns
    m3 = stats["n_gt3_dedup"].sum()
    print(f"\n   Outermost spline bin (grid={KAN_GRID_POINTS}, "
          f"bin width {bin_w:.2f}, so bin = |z| > {outer_lo:.2f}):")
    print(f"     |z| > 3 is the closest measured proxy: {m3:,} cells "
          f"({m3 / cells_d * 100:.4f}%)")
    print(f"     Clipped cells as share of that bin: "
          f"{exc_d / m3 * 100:.1f}%")
    print(f"     -> the outer bin is {100 - exc_d / m3 * 100:.1f}% genuine "
          f"unclipped variation, {exc_d / m3 * 100:.1f}% saturated.")

    # ── MASS CONCENTRATION: the number v1 asserted but never computed ─────────
    print(f"\n2. MASS CONCENTRATION  (decides per-feature fixes vs rebuild)")
    srt = stats.sort_values("n_gt5_dedup", ascending=False)
    cum = srt["n_gt5_dedup"].cumsum()
    print(f"   {'Top N features':<20} {'Exceedances':>12} {'Share of mass':>15}")
    print(f"   {'-' * 50}")
    for n in [5, 10, 20, 30, 50, 100, 200]:
        if n > len(srt):
            break
        print(f"   {'top ' + str(n):<20} {int(cum.iloc[n - 1]):>12,} "
              f"{cum.iloc[n - 1] / exc_d * 100:>14.1f}%")
    n_carry80 = int((cum / exc_d < 0.80).sum()) + 1
    n_nonzero = int((stats["n_gt5_dedup"] > 0).sum())
    print(f"\n   Features carrying 80% of all boundary mass: {n_carry80}")
    print(f"   Features with any exceedance at all:        {n_nonzero}")
    if n_carry80 <= 40:
        print(f"   -> CONCENTRATED. Fixing {n_carry80} features removes 80% "
              f"of the mass.")
    else:
        print(f"   -> DIFFUSE. Mass is spread across {n_carry80} features, so "
              f"per-feature")
        print(f"      fixes will not remove most of it. Judge on the total "
              f"({exc_d / cells_d * 100:.3f}%)")
        print(f"      instead: if the total is small, a diffuse thin tail is "
              f"harmless.")

    print(f"\n3. PER-FEATURE WALL MASS DISTRIBUTION  ({n_feat:,} features)")
    for b, n in stats["wall_mass"].map(_band).value_counts().sort_index().items():
        print(f"     {b:<14} {n:>6,}  ({n / n_feat * 100:5.1f}%)")
    for lab, m in [("Never exceed", stats["wall_mass"] == 0),
                   ("Below 0.1%", stats["wall_mass"] < 0.001),
                   ("At or above 1%", stats["wall_mass"] >= 0.01),
                   ("At or above 5%", stats["wall_mass"] >= 0.05)]:
        k = int(m.sum())
        print(f"     {lab:<18} {k:>6,} ({k / n_feat * 100:5.1f}%)")

    print(f"\n4. BY RISK CLASS")
    print(f"   'no_cross_section' = macro series, nothing averaged away. Same")
    print(f"   risk class as a raw panel feature, so this row is the best")
    print(f"   available advance estimate of panel severity.")
    print(f"\n   {'Risk class':<24} {'Feats':>6} {'Mass%':>8} {'MaxZ':>10} "
          f"{'>=1%':>6} {'>=5%':>6}")
    print(f"   {'-' * 64}")
    for rc, g in stats.groupby("risk_class"):
        print(f"   {rc:<24} {len(g):>6,} "
              f"{g['n_gt5_dedup'].sum() / g['n_obs_dedup'].sum() * 100:>8.4f} "
              f"{g['max_abs_z'].max():>10.1f} "
              f"{int((g['wall_mass'] >= 0.01).sum()):>6} "
              f"{int((g['wall_mass'] >= 0.05).sum()):>6}")

    if cfg["has_moments"]:
        print(f"\n5. BY MOMENT TYPE")
        print(f"   _spread (p90-p10) is exactly invariant to the 1/99")
        print(f"   cross-sectional winsorise and receives no upstream outlier")
        print(f"   protection at all. _cwskew/_cwkurt weight cubed and fourth")
        print(f"   powers of deviations and benefit most from it.")
        print(f"   NOTE: macro series merely ENDING in '_spread' are excluded")
        print(f"   here and counted as raw_level (v1 miscounted ~11 of them).")
        print(f"\n   {'Moment':<20} {'Feats':>6} {'Mass%':>8} {'MaxZ':>10} "
              f"{'>=1%':>6}")
        print(f"   {'-' * 54}")
        for mt in ["cwmean", "cwstd", "cwskew", "cwkurt", "spread", "raw_level"]:
            g = stats[stats["moment_type"] == mt]
            if not len(g):
                continue
            print(f"   {mt:<20} {len(g):>6,} "
                  f"{g['n_gt5_dedup'].sum() / g['n_obs_dedup'].sum() * 100:>8.4f} "
                  f"{g['max_abs_z'].max():>10.1f} "
                  f"{int((g['wall_mass'] >= 0.01).sum()):>6}")
    else:
        print(f"\n5. BY MOMENT TYPE -- not applicable")
        print(f"   means_only carries cap-weighted means under UNSUFFIXED column")
        print(f"   names, so there are no moment suffixes to group on. (v1")
        print(f"   reported 13 'spread' features here; those were macro series")
        print(f"   whose names end in the word 'spread'.)")

    print(f"\n6. BY FREQUENCY (deduped)")
    print(f"   {'Freq':<10} {'Feats':>6} {'Mass%':>8} {'MaxZ':>10} "
          f"{'MaxRunRaw':>10} {'UniqRatio':>10}")
    print(f"   {'-' * 58}")
    for fq in ["daily", "weekly", "monthly"]:
        g = stats[stats["declared_freq"] == fq]
        if not len(g):
            continue
        print(f"   {fq:<10} {len(g):>6,} "
              f"{g['n_gt5_dedup'].sum() / g['n_obs_dedup'].sum() * 100:>8.4f} "
              f"{g['max_abs_z'].max():>10.1f} {int(g['max_run_raw'].max()):>10} "
              f"{g['unique_ratio'].median():>10.3f}")
    print(f"\n   The weekly row's unique_ratio of ~1.000 is not an error: in")
    print(f"   Stage 4 these were forward-filled to daily and THEN z-scored, so")
    print(f"   a constant numerator over a moving expanding mean/std yields a")
    print(f"   new z every day. That is the original bug, measured.")

    print(f"\n7. CAUSE CLASSIFICATION")
    print(f"   {'Cause':<24} {'Feats':>6} {'ExcCells':>9} {'MassShare':>10} "
          f"{'Action':<22}")
    print(f"   {'-' * 74}")
    for cause in ["clean", "minor_tail", "artefact_spike",
                  "outlier_driven_scale", "zero_inflated",
                  "nonstationary_trend", "high_mass_unexplained"]:
        g = stats[stats["cause"] == cause]
        if not len(g):
            continue
        e = int(g["n_gt5_dedup"].sum())
        print(f"   {cause:<24} {len(g):>6,} {e:>9,} "
              f"{e / exc_d * 100:>9.1f}% {g['action'].iloc[0]:<22}")

    if stats["raw_modal_share"].notna().any():
        print(f"\n8. RAW-VALUE DIAGNOSTICS (from Stage 2, pre-z-score)")
        zi = stats[stats["raw_modal_share"] > ZERO_INFLATED_MODAL_MIN]
        print(f"   Zero-inflated (modal value >50% of obs): {len(zi)} features")
        if len(zi):
            print(f"   These are the genuine case for a denominator floor:")
            for _, r in zi.nlargest(12, "wall_mass").iterrows():
                print(f"     {r['column'][:44]:<45} modal={r['raw_modal_share']:.3f} "
                      f"zero={r['raw_zero_share']:.3f} mass={r['wall_mass'] * 100:.2f}%")

        infl = stats[stats["std_over_sigma_f"] > STD_INFLATION_MIN]
        print(f"\n   Outlier-inflated scale (std > {STD_INFLATION_MIN}x robust "
              f"scale): {len(infl)} features")
        print(f"   For these the denominator is too LARGE, so z-scores are")
        print(f"   suppressed. Any exceedance is a genuinely enormous numerator")
        print(f"   -- an upstream definition problem, not a normalisation one.")
        print(f"   A sigma_f floor would do nothing for them.")
        if len(infl):
            for _, r in infl.nlargest(12, "std_over_sigma_f").iterrows():
                print(f"     {r['column'][:44]:<45} "
                      f"std/sf={r['std_over_sigma_f']:>10.1f} "
                      f"maxz={r['max_abs_z']:>9.1f} "
                      f"mass={r['wall_mass'] * 100:.2f}%")

    if "wf_wall_mass" in stats.columns:
        wf = stats[stats["wf_wall_mass"].notna()].copy()
        if len(wf):
            print(f"\n9. WEEKLY FIX EFFECT")
            s4_daily = int(wf["n_gt5_raw"].sum())
            wf_daily = int(wf["wf_n_gt5_raw"].sum())
            print(f"   Clipped DAILY cells across the 34 weekly features:")
            print(f"     Stage 4 (daily-drifted z): {s4_daily:>6,}")
            print(f"     Weekly-frequency z:        {wf_daily:>6,}")
            print(f"   The fix removes the spurious daily drift, which is")
            print(f"   correct and worth doing, but note it does NOT")
            print(f"   materially reduce clipped mass -- each weekly")
            print(f"   exceedance is still forward-filled across ~5 days.")
            print(f"\n   {'Feature':<22} {'S4max':>8} {'WFmax':>8} "
                  f"{'S4dailyExc':>11} {'WFdailyExc':>11}")
            print(f"   {'-' * 64}")
            for _, r in wf.nlargest(10, "wf_wall_mass").iterrows():
                print(f"   {r['column']:<22} {r['max_abs_z']:>8.1f} "
                      f"{r['wf_max_abs_z']:>8.1f} {int(r['n_gt5_raw']):>11,} "
                      f"{int(r['wf_n_gt5_raw']):>11,}")

    print(f"\n10. TOP 25 BY WALL MASS")
    print(f"   {'Feature':<42} {'Mass%':>7} {'nExc':>5} {'MaxZ':>9} "
          f"{'DateOfMax':>11} {'Cause':<20}")
    print(f"   {'-' * 100}")
    for _, r in stats.head(25).iterrows():
        print(f"   {r['column'][:41]:<42} {r['wall_mass'] * 100:>7.3f} "
              f"{int(r['n_gt5_dedup']):>5} {r['max_abs_z']:>9.1f} "
              f"{r['date_of_max']:>11} {r['cause']:<20}")

    print(f"\n11. TOP 20 BY MAX |z|  (magnitude, not mass)")
    print(f"   {'Feature':<42} {'MaxZ':>10} {'DateOfMax':>11} "
          f"{'std/sf':>9} {'Cause':<20}")
    print(f"   {'-' * 96}")
    for _, r in stats.nlargest(20, "max_abs_z").iterrows():
        sf = (f"{r['std_over_sigma_f']:.1f}"
              if not np.isnan(r.get("std_over_sigma_f", np.nan)) else "n/a")
        print(f"   {r['column'][:41]:<42} {r['max_abs_z']:>10.1f} "
              f"{r['date_of_max']:>11} {sf:>9} {r['cause']:<20}")

    print(f"\n12. DATES APPEARING MOST OFTEN AS A FEATURE'S MAXIMUM")
    print(f"   A date recurring here across many unrelated features points to a")
    print(f"   single upstream event on that date rather than to per-feature")
    print(f"   pathology -- one bad tape day, or a genuine market-wide shock.")
    vc = stats[stats["max_abs_z"] > 10]["date_of_max"].value_counts().head(12)
    for d, n in vc.items():
        print(f"     {d}   {n:>4} features")

    total = exc_d / cells_d * 100
    print(f"\n{'=' * 78}")
    print(f"VERDICT: {name}")
    print(f"{'=' * 78}")
    print(f"\n  Total deduped boundary mass: {total:.4f}%")
    print(f"  Features carrying 80% of it:  {n_carry80}")
    print(f"  Clipped share of outer bin:   {exc_d / m3 * 100:.1f}%")

    if total < 0.5 and n_carry80 <= 40:
        print(f"""
  SMALL AND CONCENTRATED. Clipping at +/-{CLIP_LIMIT} is benign, and fixing
  {n_carry80} features removes 80% of what mass there is. Per-feature fixes,
  then clip.""")
    elif total < 0.5:
        print(f"""
  SMALL BUT DIFFUSE. Total mass {total:.3f}% is low enough that clipping is
  harmless, but it is spread across {n_carry80} features, so per-feature fixes
  will not remove most of it and are not worth doing for that purpose. Fix the
  features that are WRONG (artefacts, zero-inflated, trending) because they are
  wrong, not to reduce boundary mass. The outer spline bin is
  {100 - exc_d / m3 * 100:.0f}% genuine variation, so it will not be dominated
  by saturated values.""")
    elif total < 2.0:
        print(f"""
  MODERATE. {total:.3f}% total. Check section 4: if 'no_cross_section' mass far
  exceeds 'cross_section_averaged', the macro features need their own treatment
  and the panel will be worse still.""")
    else:
        print(f"""
  MATERIAL. {total:.3f}% total. Per-feature fixes will not be enough; apply the
  robust treatment uniformly and difference the trending features.""")


# ═══════════════════════════════════════════════════════════════════════════════
# MAIN
# ═══════════════════════════════════════════════════════════════════════════════

def main():
    print("=" * 78)
    print("BOUNDARY MASS DIAGNOSTIC v2 -- AGGREGATE DATASETS")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 78)
    print(f"\nUnclipped z-scores from Stage 4: {STAGE4_DIR}")
    print(f"Raw values for enrichment from:  {STAGE2_DIR}")
    print(f"Clip limit under test: +/-{CLIP_LIMIT}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 78}")
    print("WEEKLY Z-SCORE RECONSTRUCTION")
    print(f"{'=' * 78}")
    if PANEL_C_PATH.exists():
        weekly_z, n_upd = build_weekly_zscored_daily(
            PANEL_C_PATH, WEEKLY_FEATURES, H41_DELAY_FEATURES)
        u = pd.Series(n_upd)
        print(f"  {len(WEEKLY_FEATURES)} features rebuilt at weekly frequency")
        print(f"  Updates: min={u.min()}, median={u.median():.0f}, max={u.max()}")
        print(f"  Expected ~940 for 18 years. Counts look correct, so")
        print(f"  value-change detection is NOT merging identical weeks.")
    else:
        print(f"  MISSING {PANEL_C_PATH}; weekly comparison skipped")
        weekly_z = pd.DataFrame({"date": []})

    summary = {}
    for name, cfg in DATASETS.items():
        stats = analyse_dataset(name, cfg, weekly_z)
        if stats is None:
            continue
        report(name, stats, cfg)

        out = OUTPUT_DIR / f"boundary_mass_{name}.csv"
        stats.to_csv(out, index=False)
        print(f"\n  Saved: {out}")

        cd = int(stats["n_obs_dedup"].sum())
        ed = int(stats["n_gt5_dedup"].sum())
        srt = stats.sort_values("n_gt5_dedup", ascending=False)
        cum = srt["n_gt5_dedup"].cumsum()
        summary[name] = {
            "n_features": len(stats),
            "mass_pct_dedup": ed / cd * 100,
            "mass_pct_raw": (int(stats["n_gt5_raw"].sum())
                             / int(stats["n_obs_raw"].sum()) * 100),
            "features_carrying_80pct": int((cum / ed < 0.80).sum()) + 1,
            "clipped_share_of_outer_bin": ed / int(stats["n_gt3_dedup"].sum()) * 100,
            "n_clean": int((stats["wall_mass"] == 0).sum()),
            "n_ge_1pct": int((stats["wall_mass"] >= 0.01).sum()),
            "n_ge_5pct": int((stats["wall_mass"] >= 0.05).sum()),
            "max_abs_z": float(stats["max_abs_z"].max()),
            "cause_counts": stats["cause"].value_counts().to_dict(),
            "cause_mass_share": {
                c: float(g["n_gt5_dedup"].sum() / ed * 100)
                for c, g in stats.groupby("cause")},
            "mass_by_risk_class": {
                rc: float(g["n_gt5_dedup"].sum() / g["n_obs_dedup"].sum() * 100)
                for rc, g in stats.groupby("risk_class")},
        }

    with open(OUTPUT_DIR / "boundary_mass_summary.json", "w") as f:
        json.dump({
            "created": datetime.now().isoformat(),
            "version": 2,
            "clip_limit": CLIP_LIMIT,
            "datasets": summary,
        }, f, indent=2, default=str)
    print(f"\n  Saved: {OUTPUT_DIR / 'boundary_mass_summary.json'}")

    if len(summary) > 1:
        print(f"\n{'=' * 78}")
        print("CROSS-DATASET COMPARISON")
        print(f"{'=' * 78}")
        print(f"\n  {'Dataset':<16} {'Feats':>7} {'Mass%':>8} {'80%by':>7} "
              f"{'MaxZ':>11} {'>=1%':>6}")
        print(f"  {'-' * 60}")
        for n, s in summary.items():
            print(f"  {n:<16} {s['n_features']:>7,} "
                  f"{s['mass_pct_dedup']:>8.4f} "
                  f"{s['features_carrying_80pct']:>7} "
                  f"{s['max_abs_z']:>11.1f} {s['n_ge_1pct']:>6}")
        print(f"\n  If the two mass figures are near-identical, that is expected")
        print(f"  rather than suspicious: full_moments contains means_only's")
        print(f"  features plus four more moments per factor, and the moments")
        print(f"  have similar exceedance rates, so the pooled average converges.")

    print(f"\n{'=' * 78}")
    print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'=' * 78}")


if __name__ == "__main__":
    main()

BOUNDARY MASS DIAGNOSTIC v2 -- AGGREGATE DATASETS
Started: 2026-07-30 19:59:47

Unclipped z-scores from Stage 4: ..\..\Data\Data_Collection\Final\Stage_4_Final_w_Calendar_Theme_Removed
Raw values for enrichment from:  ..\..\Data\Data_Collection\Final\Stage_2
Clip limit under test: +/-5.0

WEEKLY Z-SCORE RECONSTRUCTION
  34 features rebuilt at weekly frequency
  Updates: min=930, median=966, max=966
  Expected ~940 for 18 years. Counts look correct, so
  value-change detection is NOT merging identical weeks.

DATASET: means_only
  model_market_combined_means.parquet: 4,299 rows x 717 cols
  Dates: 2007-11-30 -> 2024-12-30
    Theme CSV: ..\..\Data\Splits\themes\combined_means_theme_assignment.csv
  Features analysed: 709
  Measuring Stage 4 z-scores...
  Re-measuring 34 weekly features at weekly frequency...
  Enriching from Stage 2 raw values...
    Stage 2 raw: daily (4605, 400), monthly (218, 327)
    Matched 709 / 709 features to Stage 2 raw values

REPORT: means_only

1. GLOBAL BOU